In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
import pickle
from scipy.special import expit

#### helpers

Based on code from `https://github.com/jacobmchen/proximal_w_text`. The file `preprocessing_util.py` can be downloaded there.

In [ ]:
from preprocessing_util import *

#### read data

Source data can be download on `https://physionet.org/content/mimiciii/1.4/`.

In [ ]:
# read notes data and sort
data_dir = './source_data/mimic-iii-clinical-database-1.4/'
notes = pd.read_csv(data_dir + 'NOTEEVENTS.csv' , low_memory=False).sort_values(by='CHARTDATE')

In [ ]:
# read diagnosis data
diagnoses_icd = pd.read_csv(data_dir + 'DIAGNOSES_ICD.csv')
diagnoses_icd_key = pd.read_csv(data_dir + 'D_ICD_DIAGNOSES.csv')

In [ ]:
# read patient data
patients = pd.read_csv(data_dir + 'PATIENTS.csv')
patients['gender'] = patients['GENDER'].map({'F': 0, 'M': 1})

#### process features

notes

In [ ]:
# drop missing IDS
notes = notes.dropna(subset=['HADM_ID'])

In [ ]:
# earliest record per patient
unique_subjects = set(notes['SUBJECT_ID'])
first_hadm_ids = []

# loop over subjects
for subject in unique_subjects:
    temp = notes[notes['SUBJECT_ID'] == subject]
    first_hadm_ids.append(temp.iloc[0]['HADM_ID'])

# select
notes_subset = notes[notes['HADM_ID'].isin(first_hadm_ids)]

In [ ]:
# remove discharge summaries
output_A = notes_subset[notes_subset['CATEGORY'] != 'Discharge summary']

diagnosis

In [ ]:
# clean
diagnoses_icd_key = diagnoses_icd_key.drop(columns=['LONG_TITLE', 'ROW_ID'])
output_B = diagnoses_icd.merge(diagnoses_icd_key, how='left', on='ICD9_CODE')
output_B = output_B.dropna(subset=['SHORT_TITLE'])

In [ ]:
# top 10 most frequent diagnosis
diagnosis_names = ['Hypertension NOS','Crnry athrscl natve vssl','Atrial fibrillation','CHF NOS','DMII wo cmp nt st uncntr','Hyperlipidemia NEC/NOS',
                   'Acute kidney failure NOS','Need prphyl vc vrl hepat','NB obsrv suspct infect','Acute respiratry failure']

In [ ]:
# retain top diagnosis per hospital admission and subject ID
diagnoses_df = output_A[['HADM_ID', 'SUBJECT_ID']].drop_duplicates()

# loop over top diagnosis and admissions
for diagnosis in diagnosis_names:
    l = []
    for idx, hadm_id in enumerate(diagnoses_df["HADM_ID"]):

        # check if admission is linked with diagnosis
        subset = output_B[output_B['HADM_ID'] == hadm_id]
        subset = subset[subset['SHORT_TITLE'] == diagnosis]
        if len(subset) >= 1:
            l.append(1)
        else:
            l.append(0)
    diagnoses_df[diagnosis] = l

patients

In [ ]:
# add gender and age to the clinical notes
output_A = output_A.merge(patients.set_index('SUBJECT_ID')[['gender','DOB']], left_on='SUBJECT_ID', right_index=True, how='left')
output_A['age'] = calculate_age(output_A['CHARTDATE'], output_A['DOB'])

# filter extremes
output_A = output_A[output_A['age'] > 18]
output_A = output_A[output_A['age'] < 100]

# add diagnosis
features = output_A.merge(diagnoses_df, on=['HADM_ID', 'SUBJECT_ID'], how='left')

#### generate treatments and outcomes

In [ ]:
# select relevant columns and rename
features = features[['Crnry athrscl natve vssl', 'Atrial fibrillation', 'Hypertension NOS', 'CHF NOS', 'age', 'gender']]
features.columns = ['CORONARY_ATHERO','ATRIAL_FIBRI','HYPERTENSION','CONGESTIVE_HF', 'AGE', 'GENDER']

In [ ]:
# normalize age
age_min, age_max = features['AGE'].min(), features['AGE'].max()
features['AGE_unit'] = (features['AGE'] - age_min) / (age_max - age_min)

In [ ]:
# store as variables
x_sex  = features['GENDER'].values.astype(float)           
x_age  = features['AGE_unit'].values                       
x_hyp  = features['HYPERTENSION'].values.astype(float)
x_cor  = features['CORONARY_ATHERO'].values.astype(float)
x_afib = features['ATRIAL_FIBRI'].values.astype(float)
x_chf  = features['CONGESTIVE_HF'].values.astype(float)

In [ ]:
# generate treatment
alpha = -1.2   
temp_s = 0.85

logit_core = (alpha + 0.2*x_hyp + 0.9*x_cor + 0.8*x_afib + 0.7*x_chf + 0.3*x_age + 0.2*x_sex)

p_treat = expit(temp_s * logit_core)
rng = np.random.default_rng(42)
T = rng.binomial(1, p_treat, size=len(features)).astype(int)

In [ ]:
# cate, mu0
M0 = (0.4*x_age + 0.6*x_sex+ 0.2*x_hyp + 1.1*x_cor + 0.8*x_afib + 1.4*x_chf)
cate = (-1 + 0.25*x_cor + 0.2*x_afib + 0.3*x_chf + 0.05*x_hyp + 0.35*(x_cor * x_afib) + 0.05*(x_chf * x_hyp) - 0.1*(1.0 - x_sex) + 0.10*np.sin(np.pi * x_age))
M1 = M0 + cate

In [ ]:
# potential and observed outcomes
eps0 = rng.normal(0, 1, size=len(features))
eps1 = rng.normal(0, 1, size=len(features))
Y0 = M0 + eps0
Y1 = M0 + cate + eps1
Y = np.where(T == 0, Y0, Y1)

In [ ]:
# store in df
df = features.assign(M0=M0, M1=M1, Y0=Y0, Y1=Y1, cate=cate, T=T, Y=Y)
df = df.drop("AGE_unit", axis=1)

In [ ]:
# store
df.to_csv('./datasets/mimic_syn.csv')